In [88]:
import pandas as pd
import numpy as np
import os
import shutil
from os import listdir, makedirs
from os.path import isfile, join, exists
from shutil import rmtree
from glob import glob
from collections import defaultdict

In [102]:
data_dir = Path('.')
inventory_files = list(data_dir.glob('*-inventory.csv'))

for inv_file in inventory_files:
    prefix = inv_file.stem.replace('-inventory', '')
    supply_file = data_dir / f'{prefix}-supply.csv'
    sell_file = data_dir / f'{prefix}-sell.csv'

    if not supply_file.exists() or not sell_file.exists():
        continue

    inventory = pd.read_csv(inv_file, parse_dates=['date'])
    supply = pd.read_csv(supply_file, parse_dates=['date'])
    sales = pd.read_csv(sell_file, parse_dates=['date'], on_bad_lines='skip')

    inventory['date'] = pd.to_datetime(inventory['date'], errors='coerce')
    supply['date'] = pd.to_datetime(supply['date'], errors='coerce')
    sales['date'] = pd.to_datetime(sales['date'], errors='coerce')

    inventory.columns = ['date', 'apple', 'pen']
    supply.columns = ['date', 'item', 'quantity']
    sales.columns = ['date', 'sku_num']

    inventory['apple'] = pd.to_numeric(inventory['apple'], errors='coerce')
    inventory['pen'] = pd.to_numeric(inventory['pen'], errors='coerce')
    supply['quantity'] = pd.to_numeric(supply['quantity'], errors='coerce')
    sales['sku_num'] = pd.to_numeric(sales['sku_num'], errors='coerce')

    inventory.dropna(inplace=True)
    supply.dropna(inplace=True)
    sales.dropna(inplace=True)

    start_date = inventory['date'].min()
    end_date = max(supply['date'].max(), sales['date'].max())

    all_days = pd.date_range(start_date, end_date, freq='D')

    def pivot_daily(df):
        daily = df.groupby(['date', 'item'])['quantity'].sum().unstack(fill_value=0)
        daily = daily.reindex(all_days, fill_value=0)
        if 'apple' not in daily.columns: daily['apple'] = 0
        if 'pen' not in daily.columns: daily['pen'] = 0
        return daily[['apple', 'pen']].astype(int)

    daily_supply = pivot_daily(supply)

    daily_sales = sales.groupby('date')['sku_num'].sum().reindex(all_days, fill_value=0)
    daily_sales = pd.DataFrame({'apple': daily_sales, 'pen': 0}, index=all_days).astype(int)

    inventory = inventory.set_index('date').sort_index()
    daily_inventory = pd.DataFrame(index=all_days, columns=['apple', 'pen'], dtype='float')

    for i in range(len(inventory)):
        date = inventory.index[i]
        apple = inventory.iloc[i]['apple']
        pen = inventory.iloc[i]['pen']
        next_date = inventory.index[i + 1] if i + 1 < len(inventory) else end_date + timedelta(days=1)
        daily_inventory.loc[date:next_date - timedelta(days=1), 'apple'] = apple
        daily_inventory.loc[date:next_date - timedelta(days=1), 'pen'] = pen

    daily_inventory = daily_inventory.ffill().astype(int)

    # Финальный расчёт
    result = pd.DataFrame(index=all_days, columns=['apple', 'pen'], dtype='int')
    result.iloc[0] = daily_inventory.iloc[0]

    for i in range(1, len(result)):
        prev = result.iloc[i - 1]
        supply_today = daily_supply.iloc[i]
        sales_today = daily_sales.iloc[i]
        result.iloc[i] = (prev + supply_today - sales_today).clip(lower=0).values  # .values критично

    # Сохраняем
    result = result.reset_index().rename(columns={'index': 'date'})
    output_file = data_dir / f'{prefix}-daily_inventory.csv'
    result.to_csv(output_file, index=False)


##                                       **1. Состояние склада на каждый день**

In [103]:
all_files = glob('*-daily_inventory.csv')

df_list = []
for file in all_files:
    df = pd.read_csv(file, parse_dates=['date'])
    df_list.append(df)

# Объединяем и группируем по дате
all_data = pd.concat(df_list)
total_inventory = all_data.groupby('date', as_index=False).sum()

print(total_inventory.head())

total_inventory.to_csv('total_daily_inventory.csv', index=False)


        date    apple     pen
0 2006-01-31  47363.0  3527.0
1 2006-02-01  47363.0  3527.0
2 2006-02-02  47363.0  3527.0
3 2006-02-03  47363.0  3527.0
4 2006-02-04  47363.0  3527.0


## **2. Месячные данные о количестве сворованного товара**

In [104]:
data_dir = Path('.')
inventory_files = list(data_dir.glob('*-inventory.csv'))

def calculate_monthly_theft(inventory_df, supply_df, sales_df):
    theft_records = []
    inventory_df = inventory_df.sort_values('date').reset_index(drop=True)

    for i in range(1, len(inventory_df)):
        prev_date = inventory_df.loc[i-1, 'date']
        curr_date = inventory_df.loc[i, 'date']

        prev_apple = inventory_df.loc[i-1, 'apple']
        curr_apple = inventory_df.loc[i, 'apple']
        prev_pen = inventory_df.loc[i-1, 'pen']
        curr_pen = inventory_df.loc[i, 'pen']

        mask_supply = (supply_df['date'] >= prev_date) & (supply_df['date'] <= curr_date)
        period_supply = supply_df[mask_supply]
        supply_apple = period_supply[period_supply['item'] == 'apple']['quantity'].sum()
        supply_pen = period_supply[period_supply['item'] == 'pen']['quantity'].sum()

        mask_sales = (sales_df['date'] >= prev_date) & (sales_df['date'] <= curr_date)
        period_sales = sales_df[mask_sales]
        sales_apple = period_sales['sku_num'].sum()

        expected_apple = prev_apple + supply_apple - sales_apple
        expected_pen = prev_pen + supply_pen

        theft_apple = expected_apple - curr_apple
        theft_pen = expected_pen - curr_pen

        theft_apple = theft_apple if theft_apple > 0 else 0
        theft_pen = theft_pen if theft_pen > 0 else 0

        theft_records.append({
            'date': curr_date,
            'apple': theft_apple,
            'pen': theft_pen
        })

    return pd.DataFrame(theft_records)

for inv_file in inventory_files:
    prefix = inv_file.stem.replace('-inventory', '')
    supply_file = data_dir / f'{prefix}-supply.csv'
    sell_file = data_dir / f'{prefix}-sell.csv'

    if not supply_file.exists() or not sell_file.exists():
        continue
    inventory = pd.read_csv(inv_file, parse_dates=['date'])
    supply = pd.read_csv(supply_file, parse_dates=['date'])
    sales = pd.read_csv(sell_file, parse_dates=['date'], on_bad_lines='skip')

    inventory['date'] = pd.to_datetime(inventory['date'], errors='coerce')
    supply['date'] = pd.to_datetime(supply['date'], errors='coerce')
    sales['date'] = pd.to_datetime(sales['date'], errors='coerce')

    inventory.columns = ['date', 'apple', 'pen']
    supply.columns = ['date', 'item', 'quantity']
    sales.columns = ['date', 'sku_num']

    inventory['apple'] = pd.to_numeric(inventory['apple'], errors='coerce')
    inventory['pen'] = pd.to_numeric(inventory['pen'], errors='coerce')
    supply['quantity'] = pd.to_numeric(supply['quantity'], errors='coerce')
    sales['sku_num'] = pd.to_numeric(sales['sku_num'], errors='coerce')

    inventory.dropna(inplace=True)
    supply.dropna(inplace=True)
    sales.dropna(inplace=True)

    inventory = inventory.sort_values(by='date').reset_index(drop=True)

    theft_df = calculate_monthly_theft(inventory, supply, sales)

    output_file = data_dir / f'{prefix}-monthly_theft.csv'
    theft_df.to_csv(output_file, index=False)

all_theft_files = list(data_dir.glob('*-monthly_theft.csv'))
df_list = []
for file in all_theft_files:
    df = pd.read_csv(file, parse_dates=['date'], on_bad_lines='skip')
    df_list.append(df)

if df_list:
    all_theft = pd.concat(df_list, ignore_index=True)
    total_theft = all_theft.groupby('date', as_index=False).sum()


## **3. Агрегированные данные об объемах продаж и количестве сворованной продукции по штату и году**

In [105]:
sell_parser = lambda transaction: transaction[6]

inp_dir = "input"
out_dir = "output"
all_files = [f for f in listdir(inp_dir) if isfile(join(inp_dir, f))]
stores = [f[:-8] for f in all_files if f.endswith("sell.csv")]

if not exists(out_dir):
    makedirs(out_dir)
else:
    rmtree(out_dir)
    makedirs(out_dir)

statistics = []
def agregate_statistics(stats_list):
    all_stats = pd.concat(stats_list, ignore_index=True)
    agg = all_stats.groupby(['year', 'state']).agg({
        'apple_sold': 'sum',
        'pen_sold': 'sum',
        'apple_stolen': 'sum',
        'pen_stolen': 'sum'
    }).reset_index()
    return agg

for store in stores:

    try:
        inventory = pd.read_csv(join(inp_dir, store + "inventory.csv"))
        sold = pd.read_csv(join(inp_dir, store + "sell.csv"))
        supply = pd.read_csv(join(inp_dir, store + "supply.csv"))
    except Exception as e:
        continue

    # state  – ежедневное состояние склада,
    # stolen – месячные данные о сворованном товаре,
    # sells – данные о продажах по месяцам.
    state, stolen, sells = process_store(sold, supply, inventory, sell_parser)

    state.to_csv(join(out_dir, store + "daily.csv"), index=False)
    stolen.to_csv(join(out_dir, store + "steal.csv"), index=False)

    sells.index = stolen.index

    stats = sells.join(stolen, lsuffix='_sold', rsuffix='_stolen').reset_index()
    stats['state'] = store[:2]
    stats['year'] = stats['date'].apply(lambda d: str(d)[:4])

    statistics.append(stats)

aggr_stats = agregate_statistics(statistics)
print("📊 Агрегированные данные по продажам и кражам (по штату и году):")
print(aggr_stats)
aggr_stats.to_csv(join(out_dir, "states.csv"), index=False)


<ipython-input-93-19da8ef1598b>:29: FutureWarning: The provided callable <function cumsum at 0x7f4fd99bc680> is currently using DataFrameGroupBy.cumsum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "cumsum" instead.
  monthly_cumsum = changes_daily.groupby(lambda d: d[:7]).agg(np.cumsum)
<ipython-input-93-19da8ef1598b>:29: FutureWarning: The provided callable <function cumsum at 0x7f4fd99bc680> is currently using DataFrameGroupBy.cumsum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "cumsum" instead.
  monthly_cumsum = changes_daily.groupby(lambda d: d[:7]).agg(np.cumsum)
<ipython-input-93-19da8ef1598b>:29: FutureWarning: The provided callable <function cumsum at 0x7f4fd99bc680> is currently using DataFrameGroupBy.cumsum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "cumsum"

📊 Агрегированные данные по продажам и кражам (по штату и году):
   year state  apple_sold  pen_sold  apple_stolen  pen_stolen
0  2006    MS      789324     50523         262.0       279.0
1  2007    MS      787822     50852         241.0       194.0
2  2008    MS      793339     50533         227.0       236.0
3  2009    MS      787910     50587         271.0       262.0
4  2010    MS      788965     50323         282.0       249.0
5  2011    MS      823637     51801      -34683.0     -1597.0
6  2012    MS      790026     50642         217.0       239.0
7  2013    MS      791099     50230         213.0       264.0
8  2014    MS      788692     50417         253.0       267.0
9  2015    MS      788971     49954         216.0       241.0


<ipython-input-93-19da8ef1598b>:29: FutureWarning: The provided callable <function cumsum at 0x7f4fd99bc680> is currently using DataFrameGroupBy.cumsum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "cumsum" instead.
  monthly_cumsum = changes_daily.groupby(lambda d: d[:7]).agg(np.cumsum)
